In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))


In [ ]:
import anndata as ad
import torch
from torch.utils.data import DataLoader

from utils import get_scgpt_model
from premium_datasets import bulkDataset, collate_fn
from adapters import scGPTClassifier, train_epoch, eval_epoch


In [5]:
model_path = '../papers/scgpt/save/whole_human'
data_path = 'data_new/train.h5ad'

scgpt_model, vocab = get_scgpt_model(model_path, device='cpu')
adata = ad.read_h5ad(data_path)


/storage/scratch/2370352/my-research/papers/scgpt/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(


In [6]:
# ── SETUP & TRAINING ──────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset    = bulkDataset(adata, vocab)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, collate_fn=collate_fn, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, collate_fn=collate_fn, shuffle=False, num_workers=0)

num_classes = len(dataset.label2id)
model       = scGPTClassifier(scgpt_model, num_classes=num_classes, emb_dim=512).to(device)
optimizer   = torch.optim.AdamW(model.classifier.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

for epoch in range(20):
    train = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_epoch(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch+1:02d} | "
          f"loss {train['loss']:.4f} | "
          f"acc_main {train['acc_main']:.3f} acc_rand {train['acc_rand']:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f}")


  0%|          | 0/280 [00:00<?, ?it/s]/scratch/2370352/conda/envs/scgpt/lib/python3.11/site-packages/torch/nn/modules/transformer.py:384: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)
100%|██████████| 70/70 [01:12<00:00,  1.04s/it]


Epoch 01 | loss 1.0282 | acc_main 0.518 acc_rand 0.512 | val loss 1.4767 acc 0.538


100%|██████████| 70/70 [02:34<00:00,  2.21s/it]


Epoch 02 | loss 0.9122 | acc_main 0.533 acc_rand 0.533 | val loss 1.4349 acc 0.543


100%|██████████| 70/70 [01:15<00:00,  1.08s/it]


Epoch 03 | loss 0.8723 | acc_main 0.542 acc_rand 0.535 | val loss 1.3920 acc 0.545


 18%|█▊        | 50/280 [01:52<08:38,  2.25s/it]


KeyboardInterrupt: 